# Crop Disease Prediction
Synthetic/demo dataset for a college ML + Blockchain project. The dataset is not an official agricultural dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle


In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "ml/dataset/crop_disease.csv").exists():
    ROOT = ROOT.parent.parent
DATA_PATH = ROOT / "ml/dataset/crop_disease.csv"
MODEL_PATH = ROOT / "ml/models/crop_disease_model.pkl"
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nStatistics:\n", df.describe(include="all"))


In [ ]:
X = df.drop(columns=["Disease"])
y = df["Disease"]
categorical_features = ["Crop"]
numeric_features = ["Temperature", "Humidity", "Soil_Moisture", "Rainfall", "pH"]
preprocessor = ColumnTransformer([
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
    ("numeric", SimpleImputer(strategy="median"), numeric_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


In [ ]:
cm = confusion_matrix(y_test, pred, labels=sorted(y.unique()))
plt.figure(figsize=(12,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
sample = pd.DataFrame([{
    "Crop":"Rice", "Temperature":28, "Humidity":80, "Soil_Moisture":65, "Rainfall":120, "pH":6.5
}])
print("Sample prediction:", model.predict(sample)[0])
print("Probabilities:", dict(zip(model.classes_, model.predict_proba(sample)[0])))


In [ ]:
with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)
print("Saved model to", MODEL_PATH)
